# Занятие 6. Параллельные тексты, alignment и переводческий baseline

**Цель практики:** загрузить маленький татарско-русский параллельный корпус, проверить кандидаты выравнивания и попробовать простой multilingual embedding baseline.

Работает в бесплатном Colab на CPU. Если Hugging Face загрузка недоступна, тетрадка использует встроенный мини-набор для демонстрации.

In [ ]:
!pip -q install datasets sentence-transformers pandas scikit-learn

In [ ]:
import os, re, json, textwrap, math, statistics, random, io, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from sklearn.metrics.pairwise import cosine_similarity

## 1. Загружаем небольшой фрагмент параллельного корпуса

In [ ]:
fallback = pd.DataFrame({
    'tat': [
        'Мин мәктәпкә барам.',
        'Бу китап бик кызык.',
        'Без бүген яңа проект башлыйбыз.',
        'Телне саклау өчен мәгълүмат кирәк.',
        'Укытучы сораулар бирә.',
    ],
    'rus': [
        'Я иду в школу.',
        'Эта книга очень интересная.',
        'Сегодня мы начинаем новый проект.',
        'Для сохранения языка нужны данные.',
        'Учитель задает вопросы.',
    ],
})

try:
    from datasets import load_dataset
    ds = load_dataset('AigizK/tatar-russian-parallel-corpora', split='train[:80]')
    pairs = ds.to_pandas()
    print(pairs.columns)
    # Пытаемся найти текстовые колонки автоматически.
    text_cols = [c for c in pairs.columns if pairs[c].dtype == 'object']
    pairs = pairs[text_cols[:2]].dropna().head(50)
    pairs.columns = ['tat', 'rus']
except Exception as e:
    print('HF loading failed, using fallback:', e)
    pairs = fallback

show_df(pairs, 10)
save_artifact('lesson06_parallel_pairs.csv', pairs)

## 2. Создаем испорченный порядок и ищем alignment

In [ ]:
left = pairs['tat'].reset_index(drop=True)
right = pairs['rus'].sample(frac=1, random_state=7).reset_index(drop=True)
candidate_df = pd.DataFrame({'tat': left, 'rus_shuffled': right})
show_df(candidate_df, 10)

## 3. Embedding baseline для поиска пар

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
emb_left = model.encode(left.tolist(), normalize_embeddings=True)
emb_right = model.encode(right.tolist(), normalize_embeddings=True)
sim = cosine_similarity(emb_left, emb_right)

matches = []
for i in range(len(left)):
    j = int(sim[i].argmax())
    matches.append({
        'tat': left.iloc[i],
        'best_rus': right.iloc[j],
        'score': float(sim[i, j]),
        'gold_rus': pairs['rus'].iloc[i],
        'correct_exact': right.iloc[j] == pairs['rus'].iloc[i],
    })
matches = pd.DataFrame(matches)
show_df(matches, 20)
print('exact alignment accuracy:', matches['correct_exact'].mean())
save_artifact('lesson06_alignment_candidates.csv', matches)

## 4. Что делать с переводом

In [ ]:
low_conf = matches[matches['score'] < 0.5]
report = {
    'pairs': len(matches),
    'exact_alignment_accuracy': float(matches['correct_exact'].mean()),
    'low_confidence_pairs': len(low_conf),
    'next_steps': [
        'проверить top-20 кандидатов вручную',
        'сохранять score и источник каждой пары',
        'не обучать MT на непроверенных парах',
        'сделать dev/test из вручную подтвержденных пар',
    ],
}
print(json.dumps(report, ensure_ascii=False, indent=2))
save_artifact('lesson06_alignment_report.json', json.dumps(report, ensure_ascii=False, indent=2))

## Вопросы для отчёта

1. Какие пары baseline нашел неверно?
2. Можно ли доверять score без ручной проверки?
3. Сколько проверенных пар нужно собрать до первого MT baseline?
4. Какие права и источники нужно сохранить рядом с парами?